[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/42.%20The%20Peak%20Energy%20Consumption%20%26%20Load%20Shifting%20Problem/P42-Tier-8_executed.ipynb)

---


# 42. The Peak Energy Consumption & Load Shifting Problem

## Tier 8: The Value-Aligned and Ethical Framework

### Goal

This notebook evaluates a small, predefined set of energy-management scenarios using ethical constraints and a stakeholder-weighted score. The aim is to make trade-offs among operating costs, emissions, community welfare, worker safety, energy justice, sustainability, and grid stability explicit.

The notebook does **not** optimize dispatch decisions or calculate a load-shifting schedule. It evaluates the scenario data supplied below; its recommendation is therefore conditional on those data and the stated scoring rules.

### Inputs and outputs

- **Inputs:** stakeholder groups and weights, ethical thresholds, and five scenarios with reported cost and impact measures.
- **Process:** check each scenario against hard and soft constraints, calculate a weighted score for scenarios that satisfy the hard constraints, rank the scenarios, and visualize their reported attributes.
- **Outputs:** compliance status, recorded constraint violations, value-aligned scores, a ranking, and plots. The heat-wave demonstration is a separate illustrative example and is not calculated by the scenario evaluator.

### Scope and modeling assumptions

- Scenario impacts are supplied estimates on a 0–1 scale unless a field explicitly states a percentage or an emissions quantity. The health, safety, justice, sustainability, and stability scores are interpreted as normalized indicators; higher values are more favorable.
- Carbon emissions are measured in metric tonnes of CO₂-equivalent per year. The annual baseline is 2,850 tCO₂e, and the required reduction is 18.3%.
- `cost_reduction` and `cost_increase` are separately reported scenario measures with distinct reference bases. They are not assumed to be mathematical complements. The signed `cost_increase` value is used only in the economic-sustainability check and the operations score.
- A community-welfare score of at least 0.80 is required. The constraint threshold is stated directly in the model.
- Stakeholder weights determine the relative contribution of each group to the score. The stakeholder value profiles are contextual information; the score uses the impact measures explicitly specified in the scoring rule.
- The model does not represent hourly load profiles, facility-level demand, actual grid dispatch, uncertainty, stakeholder feedback, or the distribution of impacts within a stakeholder group. Its scores are decision-support indicators, not proof of ethical acceptability or regulatory compliance.

### Evaluation method

Let \(\mathcal{S}\) be the set of supplied scenarios and \(\mathcal{C}\) the set of constraints. For each scenario \(s\), the evaluator checks every constraint and records any violation. A hard-constraint violation makes the scenario ineligible for a positive value-aligned score:

\[
H(s) = \begin{cases}
1, & \text{if scenario } s \text{ violates at least one hard constraint},\\
0, & \text{otherwise}.
\end{cases}
\]

For a scenario without hard violations, the base score is a weighted sum of stakeholder scores:

\[
B(s) = \sum_{g \in \mathcal{G}} w_g q_g(s),
\qquad \sum_{g \in \mathcal{G}} w_g = 1.
\]

Here, \(\mathcal{G}\) is the set of stakeholder groups, \(w_g\) is group \(g\)'s weight, and \(q_g(s)\) is its scenario score as defined in `calculate_value_aligned_score`. The final score subtracts a penalty for soft violations, adds a specified vulnerability bonus when its conditions are met, and is clipped to the interval \([0,1]\):

\[
V(s) = \begin{cases}
0, & H(s)=1,\\
\min\!\left(1,\max\!\left(0, B(s)-P(s)+U(s)\right)\right), & H(s)=0.
\end{cases}
\]

In the code, `B(s)` is `base_score`, `P(s)` is `penalty`, and `U(s)` is `vulnerability_bonus`. Each soft-constraint penalty is 0.15 times that constraint's `penalty_weight`; the separate minimum-cost-reduction rule has a penalty of 0.15. A scenario meeting both the community-health and worker-safety conditions receives a bonus of 0.10. The ranking sorts scenarios by `value_aligned_score`; it does not search for new scenarios.

This evaluation takes \(O(|\mathcal{S}|(|\mathcal{C}|+|\mathcal{G}|))\) time: each scenario is checked against the constraints and scored across the stakeholder groups. The calculation is intentionally a transparent, finite-scenario assessment rather than an optimization algorithm.

### What to look for

- Whether a scenario satisfies each hard constraint, not merely whether it has a high score.
- Which soft constraints affect a scenario's score and how their penalty weights matter.
- How the stakeholder weights and the explicitly defined impact formulas shape the ranking.
- Why cost reduction, signed cost impact, and carbon emissions should be interpreted as distinct reported measures.
- How the illustrative heat-wave response protects essential services while totaling 15 MW of demand reduction.

### Illustrative heat-wave example

The demonstration describes a grid emergency requiring a 15 MW reduction. Its listed actions are 8 MW of rescheduled industrial crane operations, 2.5 MW of non-essential lighting reduction, 3 MW of delayed non-emergency EV charging, and 1.5 MW of administrative HVAC adjustment. These add to 15 MW. The example also reports protection of community services and changes in costs, safety, and stakeholder satisfaction. Those reported figures are illustrative inputs, not results computed by the scenario-scoring model.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
class EthicalPrinciple(Enum):
    """Principles used to categorize constraints in the evaluation model."""

    ENVIRONMENTAL_PROTECTION = "environmental_protection"
    ENERGY_JUSTICE = "energy_justice"
    WORKER_SAFETY = "worker_safety"
    COMMUNITY_WELFARE = "community_welfare"
    ECONOMIC_SUSTAINABILITY = "economic_sustainability"
    TRANSPARENCY = "transparency"
    GRID_STABILITY = "grid_stability"


@dataclass
class StakeholderGroup:
    """A stakeholder group, its score weight, and contextual value profile."""

    name: str
    weight: float
    values: Dict[str, float]
    impact_sensitivity: float
    vulnerability_score: float


@dataclass
class EthicalConstraint:
    """A threshold applied to one ethical principle."""

    principle: EthicalPrinciple
    threshold: float
    is_hard_constraint: bool
    description: str
    penalty_weight: float


@dataclass
class EnergyScenarioImpact:
    """Reported impact measures for an energy-management scenario."""

    cost_reduction: float  # Percentage
    carbon_emissions: float  # Metric tonnes of CO2-equivalent per year
    community_health: float  # Normalized indicator, from -1 to 1
    worker_safety: float  # Normalized indicator, from -1 to 1
    energy_justice: float  # Normalized indicator, from -1 to 1
    long_term_sustainability: float  # Normalized indicator, from -1 to 1
    grid_stability: float  # Normalized indicator, from -1 to 1


@dataclass
class EthicalEnergyScenario:
    """A named scenario together with its reported impacts and evaluation."""

    id: str
    name: str
    description: str
    cost_increase: float  # Signed percentage, as reported for this scenario
    implementation_time: str
    impact: EnergyScenarioImpact
    value_aligned_score: float
    compliance_status: str
    ethical_violations: List[str]

In [ ]:
class ValueAlignedEnergyManagement:
    """Evaluate supplied energy scenarios using ethical rules and weighted scores."""

    def __init__(self):
        self.stakeholders = self._initialize_stakeholders()
        self.carbon_budget = 2850  # Metric tonnes of CO2-equivalent per year
        self.reduction_target = 0.183
        self.minimum_cost_reduction = 5.0
        self.scenarios: List[EthicalEnergyScenario] = []
        self.constraints = self._initialize_constraints()

        weight_total = sum(stakeholder.weight for stakeholder in self.stakeholders)
        assert np.isclose(weight_total, 1.0), "Stakeholder weights must sum to 1."

    def _initialize_stakeholders(self) -> List[StakeholderGroup]:
        """Define stakeholder weights and contextual value profiles."""
        return [
            StakeholderGroup(
                name="Community",
                weight=0.30,
                values={
                    "energy_affordability": 0.9,
                    "air_quality": 0.8,
                    "health_protection": 0.9,
                    "emergency_services": 0.95,
                },
                impact_sensitivity=0.85,
                vulnerability_score=0.8,
            ),
            StakeholderGroup(
                name="Workers",
                weight=0.25,
                values={
                    "workplace_safety": 0.95,
                    "job_security": 0.8,
                    "working_conditions": 0.7,
                    "training_opportunities": 0.6,
                },
                impact_sensitivity=0.9,
                vulnerability_score=0.6,
            ),
            StakeholderGroup(
                name="Environment",
                weight=0.25,
                values={
                    "carbon_reduction": 0.9,
                    "renewable_energy": 0.8,
                    "biodiversity": 0.7,
                    "water_conservation": 0.6,
                },
                impact_sensitivity=0.7,
                vulnerability_score=0.9,
            ),
            StakeholderGroup(
                name="Operations",
                weight=0.20,
                values={
                    "operational_efficiency": 0.8,
                    "service_reliability": 0.9,
                    "cost_effectiveness": 0.7,
                    "productivity": 0.8,
                },
                impact_sensitivity=0.6,
                vulnerability_score=0.3,
            ),
        ]

    def _initialize_constraints(self) -> List[EthicalConstraint]:
        """Set non-negotiable thresholds and weighted policy thresholds."""
        carbon_limit = self.carbon_budget * (1 - self.reduction_target)
        return [
            EthicalConstraint(
                principle=EthicalPrinciple.WORKER_SAFETY,
                threshold=0.85,
                is_hard_constraint=True,
                description="Worker safety must meet the minimum required level.",
                penalty_weight=1.0,
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.COMMUNITY_WELFARE,
                threshold=0.80,
                is_hard_constraint=True,
                description="Essential community welfare must meet the protection threshold.",
                penalty_weight=1.0,
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.ENVIRONMENTAL_PROTECTION,
                threshold=carbon_limit,
                is_hard_constraint=True,
                description=(
                    f"Annual carbon emissions must not exceed {carbon_limit:.2f} "
                    "metric tonnes of CO2-equivalent."
                ),
                penalty_weight=1.0,
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.ENERGY_JUSTICE,
                threshold=0.70,
                is_hard_constraint=False,
                description="Energy allocation should be fair across demographic groups.",
                penalty_weight=0.8,
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.ECONOMIC_SUSTAINABILITY,
                threshold=0.50,
                is_hard_constraint=False,
                description="The reported cost increase should not exceed 50%.",
                penalty_weight=0.6,
            ),
            EthicalConstraint(
                principle=EthicalPrinciple.GRID_STABILITY,
                threshold=0.80,
                is_hard_constraint=False,
                description="The scenario should support grid stability and reliability.",
                penalty_weight=0.7,
            ),
        ]

    def create_ethical_scenarios(self) -> List[EthicalEnergyScenario]:
        """Create the five supplied scenarios for evaluation."""
        scenarios = [
            EthicalEnergyScenario(
                id="pure_optimization",
                name="Pure Economic Optimization",
                description="Maximum cost reduction without ethical considerations.",
                cost_increase=-12.4,
                implementation_time="immediate",
                impact=EnergyScenarioImpact(
                    cost_reduction=25.3,
                    carbon_emissions=2120,
                    community_health=-0.40,
                    worker_safety=0.68,
                    energy_justice=0.45,
                    long_term_sustainability=0.60,
                    grid_stability=0.85,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
                ethical_violations=[],
            ),
            EthicalEnergyScenario(
                id="value_aligned",
                name="Value-Aligned Optimization",
                description="Ethical optimization during an extreme-weather event.",
                cost_increase=12.4,
                implementation_time="medium_term",
                impact=EnergyScenarioImpact(
                    cost_reduction=12.4,
                    carbon_emissions=2120,
                    community_health=0.85,
                    worker_safety=0.92,
                    energy_justice=0.87,
                    long_term_sustainability=0.92,
                    grid_stability=0.88,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
                ethical_violations=[],
            ),
            EthicalEnergyScenario(
                id="community_first",
                name="Community-First Approach",
                description="Strong community protection with moderate energy goals.",
                cost_increase=18.5,
                implementation_time="long_term",
                impact=EnergyScenarioImpact(
                    cost_reduction=8.7,
                    carbon_emissions=2450,
                    community_health=0.95,
                    worker_safety=0.94,
                    energy_justice=0.92,
                    long_term_sustainability=0.78,
                    grid_stability=0.82,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
                ethical_violations=[],
            ),
            EthicalEnergyScenario(
                id="environmental_excellence",
                name="Environmental Excellence",
                description="Strong environmental performance with ethical safeguards.",
                cost_increase=22.3,
                implementation_time="long_term",
                impact=EnergyScenarioImpact(
                    cost_reduction=15.8,
                    carbon_emissions=1780,
                    community_health=0.88,
                    worker_safety=0.90,
                    energy_justice=0.85,
                    long_term_sustainability=0.95,
                    grid_stability=0.90,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
                ethical_violations=[],
            ),
            EthicalEnergyScenario(
                id="balanced_sustainability",
                name="Balanced Sustainability",
                description="A balance across the represented ethical dimensions.",
                cost_increase=15.2,
                implementation_time="medium_term",
                impact=EnergyScenarioImpact(
                    cost_reduction=18.5,
                    carbon_emissions=2320,
                    community_health=0.82,
                    worker_safety=0.88,
                    energy_justice=0.80,
                    long_term_sustainability=0.85,
                    grid_stability=0.86,
                ),
                value_aligned_score=0.0,
                compliance_status="NOT_EVALUATED",
                ethical_violations=[],
            ),
        ]
        self.scenarios = scenarios
        return scenarios

    def check_ethical_compliance(
        self, scenario: EthicalEnergyScenario
    ) -> Tuple[str, List[str]]:
        """Return the compliance status and labeled constraint violations."""
        violations = []

        for constraint in self.constraints:
            principle = constraint.principle
            threshold = constraint.threshold

            if principle == EthicalPrinciple.WORKER_SAFETY:
                value = scenario.impact.worker_safety
                violated = value < threshold
                actual = f"{value:.2f}"
                limit = f"{threshold:.2f}"
            elif principle == EthicalPrinciple.COMMUNITY_WELFARE:
                value = scenario.impact.community_health
                violated = value < threshold
                actual = f"{value:.2f}"
                limit = f"{threshold:.2f}"
            elif principle == EthicalPrinciple.ENVIRONMENTAL_PROTECTION:
                value = scenario.impact.carbon_emissions
                violated = value > threshold
                actual = f"{value:.2f} tCO2e/year"
                limit = f"{threshold:.2f} tCO2e/year"
            elif principle == EthicalPrinciple.ENERGY_JUSTICE:
                value = scenario.impact.energy_justice
                violated = value < threshold
                actual = f"{value:.2f}"
                limit = f"{threshold:.2f}"
            elif principle == EthicalPrinciple.ECONOMIC_SUSTAINABILITY:
                violated = scenario.cost_increase > threshold * 100
                actual = f"{scenario.cost_increase:.1f}%"
                limit = f"{threshold * 100:.1f}%"
            elif principle == EthicalPrinciple.GRID_STABILITY:
                value = scenario.impact.grid_stability
                violated = value < threshold
                actual = f"{value:.2f}"
                limit = f"{threshold:.2f}"
            else:
                continue

            if violated:
                violations.append(
                    f"{principle.value}: {actual} does not meet threshold {limit}."
                )

        if scenario.impact.cost_reduction < self.minimum_cost_reduction:
            violations.append(
                "minimum_cost_reduction: "
                f"{scenario.impact.cost_reduction:.1f}% is below "
                f"{self.minimum_cost_reduction:.1f}%."
            )

        status = "VIOLATION" if violations else "FULL_COMPLIANCE"
        return status, violations

    def calculate_value_aligned_score(
        self, scenario: EthicalEnergyScenario
    ) -> float:
        """Calculate the capped stakeholder-weighted score for one scenario."""
        _, violations = self.check_ethical_compliance(scenario)
        hard_principles = {
            constraint.principle.value
            for constraint in self.constraints
            if constraint.is_hard_constraint
        }
        hard_violation = any(
            violation.split(":", 1)[0] in hard_principles
            for violation in violations
        )
        if hard_violation:
            return 0.0

        stakeholder_scores = []
        for stakeholder in self.stakeholders:
            if stakeholder.name == "Community":
                score = (
                    scenario.impact.community_health * 0.4
                    + scenario.impact.energy_justice * 0.4
                    + scenario.impact.grid_stability * 0.2
                )
            elif stakeholder.name == "Workers":
                score = scenario.impact.worker_safety
            elif stakeholder.name == "Environment":
                carbon_target = self.carbon_budget * (1 - self.reduction_target)
                carbon_target_met = float(
                    scenario.impact.carbon_emissions <= carbon_target
                )
                score = (
                    carbon_target_met * 0.5
                    + scenario.impact.long_term_sustainability * 0.3
                    + scenario.impact.energy_justice * 0.2
                )
            elif stakeholder.name == "Operations":
                score = (
                    scenario.impact.cost_reduction / 25.3 * 0.5
                    + scenario.impact.grid_stability * 0.3
                    + (1 - abs(scenario.cost_increase) / 50) * 0.2
                )
            else:
                raise ValueError(f"No scoring rule is defined for {stakeholder.name}.")

            stakeholder_scores.append(score * stakeholder.weight)

        soft_penalty = 0.0
        for constraint in self.constraints:
            if constraint.is_hard_constraint:
                continue
            if any(
                violation.startswith(f"{constraint.principle.value}:")
                for violation in violations
            ):
                soft_penalty += 0.15 * constraint.penalty_weight

        if any(
            violation.startswith("minimum_cost_reduction:")
            for violation in violations
        ):
            soft_penalty += 0.15

        vulnerability_bonus = 0.0
        if (
            scenario.impact.community_health > 0.8
            and scenario.impact.worker_safety > 0.85
        ):
            vulnerability_bonus = 0.1

        base_score = sum(stakeholder_scores)
        final_score = max(0.0, base_score - soft_penalty + vulnerability_bonus)
        return min(1.0, final_score)

    def evaluate_scenarios(self) -> List[EthicalEnergyScenario]:
        """Calculate scores and compliance records for all stored scenarios."""
        for scenario in self.scenarios:
            scenario.value_aligned_score = self.calculate_value_aligned_score(
                scenario
            )
            status, violations = self.check_ethical_compliance(scenario)
            scenario.compliance_status = status
            scenario.ethical_violations = violations
        return self.scenarios


ethical_energy_system = ValueAlignedEnergyManagement()
scenarios = ethical_energy_system.create_ethical_scenarios()
scenarios = ethical_energy_system.evaluate_scenarios()

print(f"Scenarios evaluated: {len(scenarios)}")
print(
    "Stakeholder groups: "
    f"{[stakeholder.name for stakeholder in ethical_energy_system.stakeholders]}"
)
print(f"Ethical constraints: {len(ethical_energy_system.constraints)}")
print(f"Annual carbon baseline: {ethical_energy_system.carbon_budget} tCO2e/year")
print(f"Required reduction: {ethical_energy_system.reduction_target * 100:.1f}%")
print(
    "Maximum annual emissions: "
    f"{ethical_energy_system.carbon_budget * (1 - ethical_energy_system.reduction_target):.2f} tCO2e/year"
)

In [ ]:
def analyze_ethical_energy_scenarios(
    scenarios: List[EthicalEnergyScenario],
) -> Dict[str, object]:
    """Print scenario evaluations and return the score-based ranking."""
    if not scenarios:
        raise ValueError("At least one scenario is required for analysis.")

    print("\n" + "=" * 88)
    print("VALUE-ALIGNED ENERGY MANAGEMENT RESULTS")
    print("=" * 88)

    sorted_scenarios = sorted(
        scenarios, key=lambda scenario: scenario.value_aligned_score, reverse=True
    )
    print("\nSCENARIO RANKING BY VALUE-ALIGNED SCORE")
    print(f"{'Rank':<6} {'Scenario':<32} {'Score':<8} {'Compliance':<16} {'Cost reduction':<16}")
    print("-" * 88)

    for rank, scenario in enumerate(sorted_scenarios, start=1):
        print(
            f"{rank:<6} {scenario.name[:30]:<32} "
            f"{scenario.value_aligned_score:<8.3f} "
            f"{scenario.compliance_status:<16} "
            f"{scenario.impact.cost_reduction:>8.1f}%"
        )

    print("\nDETAILED RESULTS FOR THE THREE HIGHEST-SCORING SCENARIOS")
    for scenario in sorted_scenarios[:3]:
        print(f"\n{scenario.name}")
        print(f"  Description: {scenario.description}")
        print(f"  Value-aligned score: {scenario.value_aligned_score:.3f}")
        print(f"  Compliance: {scenario.compliance_status}")
        print(f"  Reported signed cost impact: {scenario.cost_increase:+.1f}%")
        print(f"  Implementation time: {scenario.implementation_time}")
        print(f"  Cost reduction: {scenario.impact.cost_reduction:.1f}%")
        print(f"  Annual emissions: {scenario.impact.carbon_emissions:.2f} tCO2e/year")
        print(f"  Community health: {scenario.impact.community_health:+.2f}")
        print(f"  Worker safety: {scenario.impact.worker_safety:+.2f}")
        print(f"  Energy justice: {scenario.impact.energy_justice:+.2f}")
        print(
            "  Long-term sustainability: "
            f"{scenario.impact.long_term_sustainability:+.2f}"
        )
        print(f"  Grid stability: {scenario.impact.grid_stability:+.2f}")
        if scenario.ethical_violations:
            print("  Recorded violations:")
            for violation in scenario.ethical_violations:
                print(f"    - {violation}")

    top_score = sorted_scenarios[0].value_aligned_score
    tied_scenarios = [
        scenario
        for scenario in sorted_scenarios
        if np.isclose(scenario.value_aligned_score, top_score, rtol=1e-9, atol=1e-12)
    ]
    print("\nSCORE-BASED RECOMMENDATION")
    if len(tied_scenarios) > 1:
        print("The highest score is tied; review these scenarios before selecting one:")
        for scenario in tied_scenarios:
            print(f"  - {scenario.name}: {scenario.value_aligned_score:.3f}")
    else:
        recommended = tied_scenarios[0]
        print(f"Highest-scoring scenario: {recommended.name}")
        print(f"  Score: {recommended.value_aligned_score:.3f}")
        print(f"  Compliance status: {recommended.compliance_status}")
        print(f"  Cost reduction: {recommended.impact.cost_reduction:.1f}%")
        print(f"  Annual emissions: {recommended.impact.carbon_emissions:.2f} tCO2e/year")

    return {
        "best_scenario": sorted_scenarios[0],
        "tied_top_scenarios": tied_scenarios,
        "all_scenarios": sorted_scenarios,
        "ethical_compliance": {
            scenario.id: scenario.compliance_status for scenario in scenarios
        },
        "value_aligned_scores": {
            scenario.id: scenario.value_aligned_score for scenario in scenarios
        },
    }


analysis_results = analyze_ethical_energy_scenarios(scenarios)

In [ ]:
def visualize_ethical_energy_analysis(
    scenarios: List[EthicalEnergyScenario], analysis_results: Dict[str, object]
) -> None:
    """Plot scenario scores, trade-offs, stakeholder impacts, and compliance."""
    if not scenarios:
        raise ValueError("At least one scenario is required for visualization.")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(
        "Value-Aligned Energy Management Analysis",
        fontsize=16,
        fontweight="bold",
    )

    # Color communicates compliance; the legend makes that mapping explicit.
    ax1 = axes[0, 0]
    names = [scenario.name for scenario in scenarios]
    scores = [scenario.value_aligned_score for scenario in scenarios]
    compliance_colors = [
        "#66a061" if scenario.compliance_status == "FULL_COMPLIANCE" else "#d87878"
        for scenario in scenarios
    ]
    bars = ax1.bar(names, scores, color=compliance_colors, edgecolor="black", alpha=0.85)
    ax1.set_xlabel("Scenario")
    ax1.set_ylabel("Value-aligned score (0–1)")
    ax1.set_title("Value-aligned scores and compliance")
    ax1.set_ylim(0, 1.08)
    ax1.grid(True, alpha=0.3, axis="y")
    ax1.tick_params(axis="x", rotation=35)
    for label in ax1.get_xticklabels():
        label.set_horizontalalignment("right")
    for bar, score in zip(bars, scores):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.015,
            f"{score:.3f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    from matplotlib.patches import Patch

    ax1.legend(
        handles=[
            Patch(facecolor="#66a061", label="Full compliance"),
            Patch(facecolor="#d87878", label="One or more violations"),
        ],
        loc="best",
    )

    # A lower vertical position represents lower annual emissions.
    ax2 = axes[0, 1]
    cost_reductions = [scenario.impact.cost_reduction for scenario in scenarios]
    emissions = [scenario.impact.carbon_emissions for scenario in scenarios]
    community_scores = [scenario.impact.community_health for scenario in scenarios]
    scatter = ax2.scatter(
        cost_reductions,
        emissions,
        c=community_scores,
        s=110,
        cmap="RdYlGn",
        vmin=-1,
        vmax=1,
        edgecolors="black",
    )
    for scenario, x_value, y_value in zip(scenarios, cost_reductions, emissions):
        ax2.annotate(
            scenario.name,
            (x_value, y_value),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax2.set_xlabel("Reported cost reduction (%)")
    ax2.set_ylabel("Annual emissions (tCO2e/year)")
    ax2.set_title("Cost reduction and annual emissions")
    ax2.grid(True, alpha=0.3)
    ax2.invert_yaxis()
    colorbar = fig.colorbar(scatter, ax=ax2)
    colorbar.set_label("Community health indicator (−1 to 1)")
    carbon_limit = ethical_energy_system.carbon_budget * (
        1 - ethical_energy_system.reduction_target
    )
    ax2.axhline(
        y=carbon_limit,
        color="blue",
        linestyle="--",
        alpha=0.8,
        label="Annual emissions limit",
    )
    ax2.legend()

    # A polar chart makes the five normalized stakeholder impact measures comparable.
    fig.delaxes(axes[1, 0])
    ax3 = fig.add_subplot(2, 2, 3, projection="polar")
    top_scenarios = sorted(
        scenarios, key=lambda scenario: scenario.value_aligned_score, reverse=True
    )[:3]
    categories = [
        "Community health",
        "Worker safety",
        "Energy justice",
        "Long-term sustainability",
        "Grid stability",
    ]
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    closed_angles = angles + angles[:1]
    colors = ["tab:blue", "tab:green", "tab:orange"]
    for color, scenario in zip(colors, top_scenarios):
        values = [
            scenario.impact.community_health,
            scenario.impact.worker_safety,
            scenario.impact.energy_justice,
            scenario.impact.long_term_sustainability,
            scenario.impact.grid_stability,
        ]
        closed_values = values + values[:1]
        ax3.plot(
            closed_angles,
            closed_values,
            marker="o",
            linewidth=2,
            label=scenario.name,
            color=color,
        )
        ax3.fill(closed_angles, closed_values, alpha=0.15, color=color)
    ax3.set_xticks(angles)
    ax3.set_xticklabels(categories, fontsize=8)
    ax3.set_ylim(-1, 1)
    ax3.set_title("Impact profiles of the three highest-scoring scenarios", pad=20)
    ax3.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)
    ax3.grid(True)

    ax4 = axes[1, 1]
    compliant = [
        scenario
        for scenario in scenarios
        if scenario.compliance_status == "FULL_COMPLIANCE"
    ]
    with_violations = [
        scenario
        for scenario in scenarios
        if scenario.compliance_status != "FULL_COMPLIANCE"
    ]
    for group, color, marker, label in [
        (compliant, "green", "o", "Full compliance"),
        (with_violations, "red", "X", "One or more violations"),
    ]:
        if group:
            ax4.scatter(
                [scenario.cost_increase for scenario in group],
                [scenario.value_aligned_score for scenario in group],
                s=100,
                color=color,
                marker=marker,
                edgecolors="black",
                alpha=0.8,
                label=label,
            )
    for scenario in scenarios:
        ax4.annotate(
            scenario.name,
            (scenario.cost_increase, scenario.value_aligned_score),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
        )
    ax4.set_xlabel("Reported signed cost impact (%)")
    ax4.set_ylabel("Value-aligned score (0–1)")
    ax4.set_title("Reported cost impact and score by compliance")
    ax4.grid(True, alpha=0.3)
    if compliant or with_violations:
        ax4.legend()

    fig.tight_layout(rect=(0, 0, 1, 0.96))
    plt.show()


visualize_ethical_energy_analysis(scenarios, analysis_results)

In [ ]:
def ethical_framework_demonstration() -> None:
    """Present the supplied heat-wave example as an illustrative case."""
    demand_reduction_mw = 15.0
    actions = [
        ("Industrial crane operations", 8.0, "rescheduled to the night shift"),
        ("Non-essential warehouse lighting", 2.5, "reduced"),
        ("Non-emergency EV charging", 3.0, "delayed by four hours"),
        ("Administrative building HVAC", 1.5, "adjusted setpoints"),
    ]
    total_reduction = sum(reduction for _, reduction, _ in actions)
    assert np.isclose(total_reduction, demand_reduction_mw)

    print("=" * 80)
    print("ILLUSTRATIVE ETHICAL RESPONSE: SEVERE HEAT-WAVE GRID EMERGENCY")
    print("=" * 80)
    print(f"Situation: Grid emergency requires {demand_reduction_mw:.0f} MW of demand reduction.")
    print("Conditions: 40°C ambient temperature; grid frequency at 49.85 Hz.")
    print("Vulnerable populations: older adults, healthcare facilities, and schools.")
    print("\nProtection priorities:")
    print("  Healthcare facilities: absolute protection")
    print("  Elderly care centers: absolute protection")
    print("  School cooling systems: high protection")
    print("  Community centers: moderate protection")
    print("\nIllustrative demand-reduction actions:")
    for name, reduction, action in actions:
        print(f"  {name}: {reduction:.1f} MW ({action})")
    print(f"  Total listed reduction: {total_reduction:.1f} MW")
    print("  Community services: 0 MW reduction (reported as fully protected)")
    print("\nOther reported illustrative outcomes:")
    print("  Low-income community impact: 2.3% increase, compared with 8.7% without constraints")
    print("  Worker safety: 9.2/10, compared with a potential 6.8/10")
    print("  Environmental justice: reported improvement following industrial load shifting")
    print("  Stakeholder satisfaction: 8.7/10, compared with 4.2/10 for an economic-only approach")
    print("  Reported additional cost: $4,800 and 12.4% relative to the stated comparison")
    print("\nInterpretation:")
    print("These figures describe the illustrative case; they are not calculated by the")
    print("scenario evaluator. A real operational plan would require validated demand,")
    print("safety, cost, emissions, and service-protection data.")


ethical_framework_demonstration()

In [ ]:
def tier_justification_analysis() -> None:
    """Explain the role, benefits, limitations, and use cases of Tier 8."""
    print("=" * 80)
    print("TIER 8: VALUE-ALIGNED AND ETHICAL ENERGY MANAGEMENT")
    print("=" * 80)

    print("\nWhy ethical evaluation matters")
    print("A cost- or efficiency-focused decision rule does not, by itself, guarantee")
    print("worker safety, community protection, fairness, or long-term sustainability.")
    print("A value-aligned framework makes selected principles and trade-offs explicit.")

    print("\nRelationship to other solution tiers")
    tier_roles = [
        ("Tier 1, mathematical formulation", "can represent ethical requirements as model constraints"),
        ("Tier 2, classic heuristic", "can apply value-based decision rules"),
        ("Tier 3, metaheuristic", "can search within value-aligned objectives and constraints"),
        ("Tier 4, AI/ML/RL", "requires safeguards, accountability, and bias assessment"),
        ("Tier 5, digital twin", "can represent ethical impacts in scenario simulations"),
        ("Tier 6, autonomous ecosystem", "requires ethical coordination across agents"),
    ]
    for tier, role in tier_roles:
        print(f"  {tier}: {role}.")

    print("\nPotential contributions")
    contributions = [
        "Explicit hard constraints for selected safety and community requirements",
        "Weighted stakeholder scoring with visible assumptions",
        "Environmental-justice and equity indicators",
        "Transparent recording of constraint violations",
        "Explicit consideration of long-term sustainability",
        "A structured basis for stakeholder review and governance",
        "A framework that can inform ESG analysis without certifying compliance",
    ]
    for contribution in contributions:
        print(f"  - {contribution}")

    print("\nPotential advantages")
    advantages = [
        ("Ethical safeguards", "hard constraints can rule out scenarios that breach specified thresholds"),
        ("Transparency", "reported scores and violations can make assumptions easier to inspect"),
        ("Stakeholder focus", "weights can represent agreed priorities when established through a legitimate process"),
        ("Risk awareness", "social and environmental indicators can complement cost measures"),
        ("Long-term perspective", "sustainability can be included alongside near-term performance"),
    ]
    for title, explanation in advantages:
        print(f"  - {title}: {explanation}.")

    print("\nLimitations and implementation challenges")
    limitations = [
        "Selecting thresholds and stakeholder weights involves value judgments.",
        "Different stakeholder priorities can conflict.",
        "A normalized score may conceal differences in impact severity or distribution.",
        "Scenario results depend on the quality and representativeness of the input data.",
        "This notebook does not implement stakeholder consultation, operational dispatch, or uncertainty analysis.",
        "Ethical scoring does not replace legal review, safety engineering, or accountable governance.",
    ]
    for limitation in limitations:
        print(f"  - {limitation}")

    print("\nAppropriate contexts")
    use_cases = [
        "High-stakes energy decisions affecting vulnerable populations",
        "Community-facing infrastructure and energy planning",
        "Long-term strategic energy and investment decisions",
        "Settings where stakeholders need to inspect explicit trade-offs",
        "Public-sector policy analysis and corporate sustainability assessment",
    ]
    for use_case in use_cases:
        print(f"  - {use_case}")

    print("\nConclusion")
    print("Ethical criteria can change which energy-management scenarios are acceptable")
    print("and how acceptable scenarios are ranked. The result is only as reliable as")
    print("its data, thresholds, scoring rules, and governance process. This framework")
    print("is a transparent scenario-evaluation example, not a validated operational system.")


tier_justification_analysis()

In [ ]:
def final_implementation_summary() -> None:
    """Summarize the evaluated scenarios and the scope of this demonstration."""
    if not scenarios:
        raise ValueError("Evaluate the scenarios before requesting the summary.")

    best_score = max(scenario.value_aligned_score for scenario in scenarios)
    top_scenarios = [
        scenario
        for scenario in scenarios
        if np.isclose(scenario.value_aligned_score, best_score, rtol=1e-9, atol=1e-12)
    ]
    compliant_scenarios = [
        scenario
        for scenario in scenarios
        if scenario.compliance_status == "FULL_COMPLIANCE"
    ]
    violating_scenarios = [
        scenario
        for scenario in scenarios
        if scenario.compliance_status != "FULL_COMPLIANCE"
    ]

    print("=" * 80)
    print("IMPLEMENTATION SUMMARY: VALUE-ALIGNED SCENARIO EVALUATION")
    print("=" * 80)
    print("\nModel components")
    print(f"  Stakeholder groups: {len(ethical_energy_system.stakeholders)}")
    print(f"  Ethical constraints: {len(ethical_energy_system.constraints)}")
    print(f"  Evaluated scenarios: {len(scenarios)}")
    print("  Scoring: weighted stakeholder indicators, soft-constraint penalties, and a conditional bonus")
    print("  Hard-constraint violations: value-aligned score set to zero")

    print("\nEvaluation summary")
    print(f"  Fully compliant scenarios: {len(compliant_scenarios)}")
    print(f"  Scenarios with one or more recorded violations: {len(violating_scenarios)}")
    if len(top_scenarios) == 1:
        best_scenario = top_scenarios[0]
        print(f"  Highest-scoring scenario: {best_scenario.name}")
        print(f"  Score: {best_scenario.value_aligned_score:.3f}")
        print(f"  Compliance: {best_scenario.compliance_status}")
        print(f"  Cost reduction: {best_scenario.impact.cost_reduction:.1f}%")
        print(f"  Annual emissions: {best_scenario.impact.carbon_emissions:.2f} tCO2e/year")
    else:
        print(f"  Highest score: {best_score:.3f} (tied)")
        for scenario in top_scenarios:
            print(f"    - {scenario.name}: {scenario.compliance_status}")

    print("\nConstraint categories")
    for constraint in ethical_energy_system.constraints:
        category = "Hard" if constraint.is_hard_constraint else "Soft"
        print(f"  {constraint.principle.value.replace('_', ' ').title()}: {category}")
        print(f"    {constraint.description}")

    value_aligned_scenario = next(
        (scenario for scenario in scenarios if scenario.id == "value_aligned"), None
    )
    if value_aligned_scenario is not None:
        carbon_reduction = (
            ethical_energy_system.carbon_budget
            - value_aligned_scenario.impact.carbon_emissions
        ) / ethical_energy_system.carbon_budget
        print("\nValue-aligned scenario measures")
        print(f"  Reported signed cost impact: {value_aligned_scenario.cost_increase:+.1f}%")
        print(f"  Reported cost reduction: {value_aligned_scenario.impact.cost_reduction:.1f}%")
        print(f"  Annual emissions: {value_aligned_scenario.impact.carbon_emissions:.2f} tCO2e/year")
        print(f"  Reduction from the annual baseline: {carbon_reduction * 100:.2f}%")
        print(f"  Compliance status: {value_aligned_scenario.compliance_status}")

    print("\nScope and interpretation")
    print("  The ranking applies only to the supplied scenarios and scoring assumptions.")
    print("  A high score is not a substitute for operational validation or stakeholder review.")
    print("  The heat-wave example is illustrative and is not generated by this evaluator.")


final_implementation_summary()